# Final OLS and bundle export

This notebook fits the final OLS model from the stable manuscript support and writes the final-model artifact tables used by the manuscript table/figure stage.

## Notebook Overview

### Purpose
Fit final OLS model on selected features and create reproducible export bundle. Export model object, coefficients, predictions, and metadata.

### Inputs
- `dfm_conditioned`: Feature DataFrame from stage 02
- `selected_features`: Feature names from stage 06
- `df_pheno`: Phenotype/outcome variable
- Bundle metadata: model info, parameters, run date, seed

### Outputs
- OLS model fit on selected features
- Export bundle: `outputs/07_ols_bundle/` containing:
  - `model.pkl`: Serialized fitted model
  - `coefficients.csv`: Model coefficients with names and SEs
  - `predictions.csv`: OLS predictions and residuals for all samples
  - `model_summary.txt`: R², adjusted-R², F-test, diagnostics
  - `metadata.json`: Run parameters, feature count, sample count, seed
- Stage 7 handoff bundle ready for manuscript tables/figures

### Parameters & Configuration
- **Model type**: OLS (default), GLM, or robust regression. OLS: standard linear regression.
- **Confidence level**: For coefficient CIs. Default 95%.
- **Diagnostics**: Boolean. If True, computes and exports residual diagnostics (Shapiro-Wilk test, Q-Q plot coords).
- **Export format**: 'pkl' (default, Python pickle), 'joblib', or 'onnx' (for interoperability).

### Expected Output
Notebook completes in ~5-15 seconds.
- OLS model fit successfully
- R² typically 0.3-0.7 (depends on signal strength and feature set quality)
- Coefficients exported: all features have non-NaN estimates and SEs
- Predictions exported: mean residual ≈ 0, std(residuals) matches model
- Bundle metadata JSON valid and complete

---


In [ ]:
import os
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    env_root = os.environ.get("BSM_PUBLIC_RF_REPO_ROOT")
    if env_root:
        candidate = Path(env_root).resolve()
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    raise RuntimeError("Could not locate the bsm-public-rf repository root.")


REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from bsm_rfm import build_manuscript_notebook_context, run_final_manuscript_artifacts_stage

context = build_manuscript_notebook_context(REPO_ROOT, "07_final_ols_and_bundle_export.ipynb")
result = run_final_manuscript_artifacts_stage(context)
result.final_artifacts.final_ols_summary

### Step 1: Fit OLS Model

Fit final ordinary least squares (OLS) regression on selected features.

**Process:**
1. Create design matrix: selected features
2. Fit OLS: min ||y - X*beta||^2
3. Compute standard errors, t-stats, p-values
4. Compute diagnostics: R², adjusted-R², F-test, residuals

**Output:**
- `ols_model`: Fitted OLS model object (statsmodels, sklearn, or custom)
- `coefficients_df`: Beta estimates, SEs, t-stats, p-values, 95% CIs

### Step 2: Export Bundle

Package model and metadata into reproducible export bundle.

**Bundle contents:** `outputs/07_ols_bundle/`
- `model.pkl`: Serialized fitted model (pickle format)
- `coefficients.csv`: Model coefficients with metadata
- `predictions.csv`: Predicted values and residuals for all samples
- `model_summary.txt`: R², model diagnostics
- `metadata.json`: Reproducibility info (run date, seed, n_samples, n_features)

**Purpose:**
- Enable downstream stages to generate tables/figures without re-fitting
- Archive reproducible model and predictions